# Aula 03 — Cadeias: o LLM que PEDE ferramenta (mas não executa)

**Objetivo:** entender a ponte entre grafo puro (Aula 02) e agente (Aula 05).
Ao final você deve saber: o que é `tool_calls`, por que `bind_tools` **não**
executa nada, e por que lista de mensagens sem `add_messages` perde histórico.

**Pré-requisitos:** Aula 01 (LLM, `content` vazio em modelo de raciocínio) e
Aula 02 (nó devolve update parcial, overwrite).

**Como usar:** rode em ordem. Autocontido (1 arquivo no Colab). Comece pela
CÉLULA 0. Cole sua chave na CÉLULA 1 (só precisa no Colab/sem .env).
Com `GROQ_API_KEY` usa Groq real; sem chave, Fake offline.


In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

In [ ]:
# CÉLULA 1 — chave do Groq (opcional: sem ela, tudo roda no Fake offline)
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir (uso local); no Colab não há .env

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).


## 1. Analogia: conversa como lista de eventos

Pense num chat de atendimento como um **log de eventos**: cada linha diz
*quem falou* e *o quê*. O atendente lê o log inteiro antes de responder.

| Chat da lanchonete | No LangChain chama | Papel |
|---|---|---|
| 'quero um pastel' (cliente) | **HumanMessage** | o que o humano disse |
| 'são R$ 8' (atendente) | **AIMessage** | o que o modelo respondeu |
| 'seja breve e educado' (manual) | **SystemMessage** | instrução fixa |
| 'total = 8' (caixa) | **ToolMessage** | resultado de ferramenta executada |

O **estado** do grafo agora é essa lista. Cada nó novo **anexa** linhas ao
log — e é por isso que o `overwrite` da Aula 02 vira um problema aqui.

In [ ]:
# CÉLULA 2 — os 4 tipos de mensagem (100% offline)
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

log = [
    SystemMessage(content='Você é o atendente da lanchonete. Seja breve.'),
    HumanMessage(content='quero um pastel'),
    AIMessage(content='são R$ 8'),
    ToolMessage(content='8', tool_call_id='call_1'),
]
for m in log:
    print('[' + type(m).__name__ + ']', repr(m.content))

log[1].pretty_print()
log[2].pretty_print()

## 2. Conceito: `bind_tools` é um cardápio, `tool_calls` é o pedido

- **`bind_tools([multiply, soma])`** não executa nada. Ele anexa à chamada do
  LLM a *descrição* das funções (nome, docstring, tipos dos args) — como entregar
  um cardápio ao cliente. O LLM decide: respondo com texto **ou** devolvo um pedido.
- **`tool_calls`** é um JSON de intenção: lista de `{name, args, id, type}`.
  Repare no que ele **não** tem: resultado. `multiply` nunca rodou — o `12` não
  existe em lugar nenhum ainda. A execução é trabalho da Aula 04 (`ToolNode`).
- **A docstring vira schema.** O LLM só sabe o que `multiply(a, b)` faz porque
  leu a docstring. Docstring ruim = LLM confuso = args errados.
- Com o `gpt-oss-20b`, pedido de tool vem com **`content` vazio** (Aula 01,
  seção 3.1): intenção não é texto, é estrutura.

In [ ]:
# CÉLULA 3 — o LLM pede a ferramenta (Groq real com chave, Fake sem)
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()
MODELO = 'openai/gpt-oss-20b'

def multiply(a: int, b: int) -> int:
    """Multiplica a por b."""
    return a * b

def soma(a: int, b: int) -> int:
    """Soma a e b."""
    return a + b

def get_llm():
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> Fake offline.')
        else:
            return ChatGroq(model=MODELO, temperature=0,
                            reasoning_effort='low').bind_tools([multiply, soma])
    from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
    print('AVISO: modo Fake offline.')
    fake = FakeMessagesListChatModel(responses=[AIMessage(
        content='',
        tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 4},
                     'id': 'call_1', 'type': 'tool_call'}])])
    return fake

llm = get_llm()
pedido = llm.invoke([HumanMessage(content='Quanto é 3 vezes 4?')])
print('content:', repr(pedido.content))
print('tool_calls:', pedido.tool_calls)
print('id do pedido:', pedido.tool_calls[0]['id'] if pedido.tool_calls else None)

## 3. Dissecando o `tool_calls`

- `name='multiply'` — **qual** função do cardápio (igual ao nome Python).
- `args={'a': 3, 'b': 4}` — argumentos **extraídos da frase** pelo LLM.
- `id='call_...'` — identificador único do pedido. Na Aula 04 o `ToolMessage`
  de resposta carrega esse `id` para o LLM saber *qual* pedido foi atendido.
- `type='tool_call'` — marcador de formato.
- `content=''` — vazio de propósito: quando há pedido, não há texto.
  (No `gpt-oss` isso é regra; nunca confie em `content` quando há `tool_calls`.)

E o mais importante: **`multiply(3, 4)` NÃO rodou**. Prove: o `12` não aparece
em nenhuma variável. `bind_tools` detecta intenção; `ToolNode` (Aula 04) executa.

In [ ]:
# CÉLULA 4 — resposta direta: sem conta, sem pedido
import os
resposta = llm.invoke([HumanMessage(content='Oi, tudo bem?')])
print('content:', repr(resposta.content))
print('tool_calls:', resposta.tool_calls)
if os.getenv('GROQ_API_KEY'):
    assert not resposta.tool_calls, 'conversa simples não deve pedir tool'
    print('OK — o LLM respondeu com texto e IGNOROU o cardápio.')
else:
    print('(no Fake offline o tool_call é sempre o mesmo gravado;')
    print(' com a chave Groq, esta célula valida resposta direta sem tool.)')


## 4. O problema: lista de mensagens SEM reducer perde histórico

Volte à Aula 02: *nó devolve update parcial, e por padrão a chave é
sobrescrita*. Com `messages` isso é fatal: o nó 2 devolve `[resposta]` e
**apaga** `[pergunta]` do nó 1. O bot 'esquece' o que o usuário disse
dentro da **mesma** execução. A célula abaixo prova (sem LLM, offline):

In [ ]:
# CÉLULA 5 — SEM reducer: o 2º nó apaga o 1º (rode e confira)
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class SemReducer(TypedDict):
    messages: list          # lista comum: overwrite puro

def no_pergunta(state: SemReducer) -> dict:
    return {'messages': [HumanMessage(content='pergunta do usuário')]}

def no_resposta(state: SemReducer) -> dict:
    return {'messages': [AIMessage(content='resposta do bot')]}

b = StateGraph(SemReducer)
b.add_node('n1', no_pergunta); b.add_node('n2', no_resposta)
b.add_edge(START, 'n1'); b.add_edge('n1', 'n2'); b.add_edge('n2', END)
out = b.compile().invoke({'messages': []})
print([(type(m).__name__, m.content) for m in out['messages']])
assert [m.content for m in out['messages']] == ['resposta do bot']
print('PROVADO: a pergunta sumiu. É esse bug que o add_messages resolve.')

## 5. A solução: `add_messages` + `MessagesState`

- **`add_messages(atual, nova)`** é um *reducer*: em vez de substituir, ele
  **concatena**. É o `operator.add` especializado para mensagens (na Aula 08
  você verá reducers genéricos e os poderes extras: sobrescrever por `id`
  e apagar com `RemoveMessage`).
- **`Annotated[list[AnyMessage], add_messages]`** diz ao grafo: 'nesta chave,
  some em vez de sobrescrever'.
- **`MessagesState`** é o `TypedDict` pré-construído que já traz isso.
  Use sempre que o estado for conversa — não reinvente.
- Bônus: `invoke({'messages': HumanMessage(...)})` com mensagem **avulsa**
  (fora de lista) também funciona — o reducer converte.

In [ ]:
# CÉLULA 6 — COM reducer: histórico preservado (+ mensagem avulsa)
from langgraph.graph import MessagesState

def m_pergunta(state: MessagesState) -> dict:
    return {'messages': [HumanMessage(content='pergunta do usuário')]}

def m_resposta(state: MessagesState) -> dict:
    return {'messages': [AIMessage(content='resposta do bot')]}

b2 = StateGraph(MessagesState)
b2.add_node('n1', m_pergunta); b2.add_node('n2', m_resposta)
b2.add_edge(START, 'n1'); b2.add_edge('n1', 'n2'); b2.add_edge('n2', END)
out2 = b2.compile().invoke({'messages': []})
print([(type(m).__name__, m.content) for m in out2['messages']])
assert [m.content for m in out2['messages']] == ['pergunta do usuário', 'resposta do bot']

# mensagem avulsa (fora de lista) também entra:
out3 = b2.compile().invoke({'messages': HumanMessage(content='Olá!')})
print([(type(m).__name__, m.content) for m in out3['messages']])
print('OK — reducer acumula; avulsa é convertida.')

## 6. O grafo da aula: um nó que chama o LLM

Juntando tudo: `MessagesState` + um nó que invoca o LLM com cardápio.
Arestas fixas `START -> no_atendente -> END` — **sem condicional**.
Por isso nada executa: não há `ToolNode` nem rota para ele.

In [ ]:
# CÉLULA 7 — o grafo de 1 nó (detecta, não executa)
from langgraph.graph import MessagesState, StateGraph, START, END

def no_atendente(state: MessagesState) -> dict:
    return {'messages': [llm.invoke(state['messages'])]}

bg = StateGraph(MessagesState)
bg.add_node('no_atendente', no_atendente)
bg.add_edge(START, 'no_atendente')
bg.add_edge('no_atendente', END)
grafo_cadeia = bg.compile()

for pergunta in ('Quanto é 3 vezes 4?', 'Oi, tudo bem?'):
    ultima = grafo_cadeia.invoke({'messages': [HumanMessage(content=pergunta)]})['messages'][-1]
    print('P:', pergunta)
    print('  content:', repr(ultima.content), '| tool_calls:', ultima.tool_calls)

## 7. Erros clássicos

1. **Achar que `bind_tools` executa.** Não executa — o `12` nunca foi calculado.
   Prova: procure `12` nas variáveis. Detecção é diferente de execução.
2. **Esquecer o `add_messages`.** Estado `messages: list` puro perde histórico
   a cada nó (CÉLULA 5). Sempre `MessagesState` para conversa.
3. **Confiar que `content` vem preenchido com `tool_calls`.** No `gpt-oss` vem
   vazio. Leia `tool_calls`, não `content`, para decidir.
4. **Docstring vaga na tool.** O LLM monta os `args` a partir dela. Sem tipos
   (`a: int`) e descrição, ele inventa argumentos.
5. **Esperar N pedidos encadeados sem loop.** O LLM pede 1 tool por resposta;
   encadear exige voltar ao LLM (Aula 05, ReAct).
6. **`temperature` alta com tools.** Com `0.9`, os `args` variam entre execuções.
   Tools pedem `temperature=0`.

## 8. Exercício guiado — calculadora que detecta (sem calcular)

Situação: a lanchonete tem **preço por item** e **taxa de entrega**. O grafo deve
detectar *qual* conta o cliente pediu: `soma` (itens + taxa) ou `multiply`
(quantidade × preço). Você só observa os `tool_calls` — nada é executado.

In [ ]:
# CÉLULA 8 — exercício guiado (rode e interprete)
def detecta_intencao(frase: str):
    ultima = grafo_cadeia.invoke({'messages': [HumanMessage(content=frase)]})['messages'][-1]
    if ultima.tool_calls:
        chamada = ultima.tool_calls[0]
        return chamada['name'], chamada['args']
    return 'resposta_direta', {'texto': ultima.content}

# Com Groq real: o LLM escolhe a tool certa. No Fake: sempre multiply(3,4).
print(detecta_intencao('Quanto é 3 vezes 4?'))
print(detecta_intencao('Oi, tudo bem?'))
# DESAFIO: com a chave Groq, teste 'some 10 com 5' e confira ('soma', {'a': 10, 'b': 5})

## 9. Desafio — cardápio de 3 tools

Adicione `dividir(a, b)` ao cardápio (`bind_tools([multiply, soma, dividir])`)
e faça 3 perguntas, uma para cada tool. Confira os `tool_calls`:

- 'Quanto é 20 dividido por 4?' vira `dividir` com `{'a': 20, 'b': 4}`
- 'some 10 com 5' vira `soma`
- 'Quanto é 3 vezes 4?' vira `multiply`

**Pegadinha (pense antes de rodar):** no modo Fake offline, o que acontece?
O Fake devolve sempre a mesma resposta gravada — ele **não escolhe**. Por isso
o desafio exige a chave Groq real. Fake serve para mecânica de grafo, não
para inteligência do LLM.

## 10. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que `bind_tools` faz — e o que ele **não** faz?
- [ ] O que há dentro de um `tool_calls` (`name`, `args`, `id`) e para que serve cada campo?
- [ ] Por que `messages: list` puro perde histórico e o que `add_messages` muda?
- [ ] O que o `MessagesState` pré-construído traz de graça?
- [ ] Por que pedido de tool vem com `content` vazio no `gpt-oss`?
- [ ] Onde o `12` (resultado) está neste grafo? (Resposta: em lugar nenhum — Aula 04.)

**Próxima aula (04 — Roteadores):** `ToolNode` executa de verdade,
`tools_condition` decide `tools` ou `END`, e o `ToolMessage` entra no histórico.